# 🎬 VIDEO-FREE · 3. Tomas de video con IA (Wan)
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivanherrerabaquero-dev/VIDEO-FREE/blob/main/colab/3_video_ia.ipynb)

Genera **tomas de video realistas** desde un texto o desde una imagen (por ejemplo, la hoja de tu personaje o la foto del auto)
y las pone en las escenas de tu proyecto. Es lo más parecido a Higgsfield que puedes correr tú mismo.

| GPU | Modelo | Calidad | Imagen → video |
|---|---|---|---|
| A100 / L4 (≥ 22 GB) | Wan 2.2 TI2V-5B | 720p, 24 fps | Sí |
| T4 gratis (16 GB) | Wan 2.1 T2V-1.3B | 480p, 16 fps | No (solo texto) |

- **Licencia:** Apache 2.0 (Wan 2.1 y 2.2).
- **Tiempo aproximado:** 5 s de video tarda ~10–20 min en L4/A100 y ~15–30 min en T4.
- Cada toma dura **unos 5 segundos**; el portal la repite si la escena es más larga.
- **Cómo marcar escenas:** en el portal, en los detalles de una escena, agrega por ejemplo
`"ia": {"prompt": "cinematic drone shot of a black 1969 convertible on a desert highway at sunset", "imagen": "medios/personajes/actor_melancolico.png"}`.
Si una escena no tiene `ia`, este cuaderno le escribe una descripción automática según su tipo.

**Antes de empezar (una sola vez):**
1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU.*
2. Crea un token de GitHub (*Settings → Developer settings → Fine-grained tokens*) con acceso solo a **VIDEO-FREE** y permiso **Contents: Read and write**.
3. En Colab, abre el icono de **llave (Secretos)**, crea `GITHUB_TOKEN`, pega el token y activa *Acceso al cuaderno*.



In [ ]:
#@title 1 · Preparar Colab y conectar con tu repo
REPO = "ivanherrerabaquero-dev/VIDEO-FREE"  #@param {type:"string"}
RAMA = "main"  #@param {type:"string"}
#@markdown Guarda copias en tu Google Drive (Colab te pedirá permiso para acceder a Drive).
USAR_DRIVE = True  #@param {type:"boolean"}
import os, sys, subprocess
if os.path.exists("/content/VIDEO-FREE/.git"):
    subprocess.run(["git", "-C", "/content/VIDEO-FREE", "pull", "-q"], check=False)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", RAMA, f"https://github.com/{REPO}", "/content/VIDEO-FREE"], check=True)
sys.path.insert(0, "/content/VIDEO-FREE/colab")
import importlib, comun; importlib.reload(comun)
info = comun.preparar(REPO, RAMA, usar_drive=USAR_DRIVE)
VRAM = info["gpu"].get("vram_gb", 0)

In [ ]:
#@title 2 · Instalar
!pip install -q -U "diffusers>=0.43" transformers accelerate ftfy imageio imageio-ffmpeg
print("Listo")

In [ ]:
#@title 3 · ¿Qué tomas genero?
PROYECTO = "carretera-sin-ti.json"  #@param {type:"string"}
QUE = "escenas con ia" #@param ["escenas con ia", "todas las escenas", "una toma suelta"]
#@markdown Para «una toma suelta»:
PROMPT = "cinematic music video shot, a young latin man with a dark beard drives a black 1969 convertible on an empty desert highway at golden hour, drone shot following the car"  #@param {type:"string"}
IMAGEN = ""  #@param {type:"string"}
#@markdown `IMAGEN`: ruta dentro del repo (ej. `medios/personajes/actor_canta.png`) para animar esa imagen. Requiere GPU de 22 GB o más.
FORMATO = "horizontal"  #@param ["horizontal", "vertical"]
PASOS = 30  #@param {type:"slider", min:15, max:50, step:5}
SEMILLA = 11  #@param {type:"integer"}

p = comun.proyecto(PROYECTO)
ESTILO = "cinematic music video, realistic, film grain, anamorphic lens, shallow depth of field, smooth camera movement"
def hora_texto(h):
    h = (h or [0.3, 0.5])[0]
    return "golden hour afternoon" if h < 0.4 else "sunset" if h < 0.6 else "twilight" if h < 0.8 else "night under the stars"
AUTOMATICO = {
    "titulo": lambda e: f"side tracking shot of a black 1969 convertible muscle car driving on a desert highway, {hora_texto(e.get('hora'))}",
    "auto": lambda e: f"side tracking shot of a black 1969 convertible driving on an empty desert highway, mountains passing, {hora_texto(e.get('hora'))}",
    "dron": lambda e: f"aerial drone shot following a black 1969 convertible on an endless desert highway, {hora_texto(e.get('hora'))}",
    "carretera": lambda e: f"close-up of a melancholic young latin man with a dark beard driving a convertible, wind in his hair, {hora_texto(e.get('hora'))}",
    "recuerdo": lambda e: "warm hazy memory, a smiling young woman with long chestnut hair in the passenger seat of a convertible, golden light, dreamy",
    "cierre": lambda e: "wide shot from behind, a black convertible stopped on the shoulder of a desert highway at night, a man standing beside it looking at the stars",
}
tomas = []  # (índice de escena o None, prompt, imagen)
if QUE == "una toma suelta":
    tomas.append((None, PROMPT, IMAGEN.strip() or None))
else:
    for i, e in enumerate(p["escenas"]):
        ia = e.get("ia")
        if ia or (QUE == "todas las escenas" and e["tipo"] in AUTOMATICO):
            prompt = (ia or {}).get("prompt") or AUTOMATICO[e["tipo"]](e)
            tomas.append((i, prompt, (ia or {}).get("imagen")))
for t in tomas:
    print(f"· escena {t[0] if t[0] is not None else '-'}: {t[1][:110]}{' (desde imagen ' + t[2] + ')' if t[2] else ''}")
print(f"\n{len(tomas)} toma(s) por generar")

In [ ]:
#@title 4 · Generar las tomas
import torch, numpy as np
from PIL import Image
from diffusers import WanPipeline, WanImageToVideoPipeline, AutoencoderKLWan
from diffusers.utils import export_to_video

GRANDE = VRAM >= 22
MODELO = "Wan-AI/Wan2.2-TI2V-5B-Diffusers" if GRANDE else "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
if GRANDE:
    ancho, alto, cuadros, fps = (1280, 704, 121, 24) if FORMATO == "horizontal" else (704, 1280, 121, 24)
else:
    ancho, alto, cuadros, fps = (832, 480, 81, 16) if FORMATO == "horizontal" else (480, 832, 81, 16)
NEGATIVO = ("static, blurry, low quality, worst quality, jpeg artifacts, deformed, disfigured, extra fingers, poorly drawn hands, "
            "poorly drawn face, distorted car, text, subtitles, watermark, logo, overexposed, gray")
print(f"Modelo: {MODELO} · {ancho}x{alto} · {cuadros} cuadros a {fps} fps")

if "pipe_t2v" not in globals():
    vae = AutoencoderKLWan.from_pretrained(MODELO, subfolder="vae", torch_dtype=torch.float32)
    pipe_t2v = WanPipeline.from_pretrained(MODELO, vae=vae, torch_dtype=dtype)
    pipe_t2v.enable_model_cpu_offload()
    pipe_i2v = None
    if GRANDE:  # el mismo modelo sirve para animar una imagen
        pipe_i2v = WanImageToVideoPipeline.from_pretrained(MODELO, vae=vae, transformer=pipe_t2v.transformer, text_encoder=pipe_t2v.text_encoder,
                                                           tokenizer=pipe_t2v.tokenizer, scheduler=pipe_t2v.scheduler, torch_dtype=dtype)
        pipe_i2v.enable_model_cpu_offload()

def preparar_imagen(camino):
    img = Image.open(comun.bajar(camino)).convert("RGB")
    k = max(ancho / img.width, alto / img.height)  # recorta al centro para llenar el cuadro
    img = img.resize((round(img.width * k), round(img.height * k)), Image.LANCZOS)
    x, y = (img.width - ancho) // 2, (img.height - alto) // 2
    return img.crop((x, y, x + ancho, y + alto))

resultados = []
for n, (i, prompt, imagen) in enumerate(tomas):
    texto = f"{prompt}, {ESTILO}"
    gen = torch.Generator(device="cuda").manual_seed(SEMILLA + n)
    if imagen and pipe_i2v is not None:
        cuadros_v = pipe_i2v(image=preparar_imagen(imagen), prompt=texto, negative_prompt=NEGATIVO, height=alto, width=ancho,
                             num_frames=cuadros, num_inference_steps=PASOS, guidance_scale=5.0, generator=gen).frames[0]
    else:
        if imagen: print("Esta GPU no puede animar imágenes; genero la toma solo con el texto.")
        cuadros_v = pipe_t2v(prompt=texto, negative_prompt=NEGATIVO, height=alto, width=ancho, num_frames=cuadros,
                             num_inference_steps=PASOS, guidance_scale=5.0, generator=gen).frames[0]
    crudo = comun.TRABAJO / f"toma_{n}_crudo.mp4"
    export_to_video(cuadros_v, str(crudo), fps=fps)
    final = comun.TRABAJO / f"toma_{n}.mp4"  # H.264 compatible con cualquier navegador
    comun.sh(f'ffmpeg -hide_banner -loglevel error -y -i "{crudo}" -c:v libx264 -crf 20 -preset medium -pix_fmt yuv420p -movflags +faststart -an "{final}"')
    resultados.append((i, prompt, imagen, final))
    print(f"Toma {n + 1}/{len(tomas)} lista"); comun.ver(final)

In [ ]:
#@title 5 · Guardar las tomas en tu repo y ponerlas en las escenas
from pathlib import Path
nombre = comun.slug(p.get("titulo", "video"))
for n, (i, prompt, imagen, final) in enumerate(resultados):
    destino = f"medios/escenas/{nombre}_{'escena' + str(i) if i is not None else 'toma'}_{SEMILLA + n}.mp4"
    comun.a_drive(final, Path(destino).name)
    comun.guardar(destino, final.read_bytes(), f"Toma IA: {prompt[:60]}")
    if i is not None:
        e = p["escenas"][i]
        antes = e.get("ia") or {}
        e["ia"] = {**antes, "prompt": prompt, **({"imagen": imagen} if imagen else {}), "tipoOriginal": antes.get("tipoOriginal", e["tipo"]), "clip": destino}
        e["tipo"], e["archivo"] = "video", destino
if any(i is not None for i, *_ in resultados):
    comun.guardar_proyecto(PROYECTO, p, f"Proyecto: tomas IA para {p.get('titulo')}")
print("Listo. En el portal pulsa «Recargar proyecto» para ver las tomas en sus escenas.")